# Gait-ViViT: A Video Processing Model for Parkinson's Disease Detection

In [ ]:
# Required libraries.
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 5 - Performance Analysis

After training each version of the model and re-organizing the files, the final step consists of understanding how well the model performs on the task, focusing on the **F1-Score** metric due to the class imbalance of the dataset.

### Testing Metrics

Analysing the testing metrics, as well as their distribution across folds, allows to understand how much each version of the model is able to generalize on unseen data.

In [ ]:
def test_results(i=0, few_shot=False):
  if few_shot:
    # Load the metrics associated to the few-shot model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/few_shot/model_test_1_fold_{k + 1}.csv") for k in range(5)]
  else:
    # Load the metrics associated to one version of the transformer model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/v{i + 1}/model_test_{i + 1}_fold_{k + 1}.csv") for k in range(5)]
  all_data = pd.concat(files, ignore_index=True)

  # Compute mean and standard deviation for precision, recall and F1.
  precision_mean, precision_std = all_data["precision"].mean(), all_data["precision"].std()
  recall_mean, recall_std = all_data["recall"].mean(), all_data["recall"].std()
  f1_mean, f1_std = all_data["f1"].mean(), all_data["f1"].std()

  return {"precision": (precision_mean, precision_std),
          "recall": (recall_mean, recall_std),
          "f1": (f1_mean, f1_std)}

In [ ]:
# Run the function once on the few-shot model.
print("--- Few-Shot Model ---")
values = test_results(i=0, few_shot=True)
print(f"Precision: {values['precision']}")
print(f"Recall: {values['recall']}")
print(f"F1: {values['f1']}")

# Iterate through each version of the transformer model.
print("--- Transformer Model ---")
for i in range(6):
  print(f"--- Version {i + 1} ---")
  values = test_results(i=i, few_shot=False)
  print(f"Precision: {values['precision']}")
  print(f"Recall: {values['recall']}")
  print(f"F1: {values['f1']}")

### Training and Validation Performance

Measuring how training and validation performance varies throughout the fine-tuning phase can come in handy to spot eventual patterns in early stopping triggers or hyperparameter combinations.

In [ ]:
def metric_plot(mode, metric="f1", i=0, few_shot=False, plot_path="/content/drive/MyDrive/bachelor_thesis/results/performance.png"):
  if few_shot:
    # Load the metrics associated to the few-shot model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/few_shot/model_ft_1_fold_{k + 1}.csv") for k in range(5)]
  else:
    # Load the metrics associated to one version of the transformer model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/v{i + 1}/model_ft_{i + 1}_fold_{k + 1}.csv") for k in range(5)]
  all_data = pd.concat(files, ignore_index=True)

  # Create a new column to identify hyperparameter combinations.
  all_data["params"] = all_data.apply(lambda r: f"lr={r["start_lr"]}, wd={r["weight_decay"]}", axis=1)

  # Create the line plot, using the errorbar="sd" argument to plot mean and standard deviation.
  plt.figure(figsize=(10, 6))
  sns.lineplot(
      data=all_data,
      x="epoch",
      y=metric,
      hue="params",
      style="params",
      errorbar="sd")
  plt.title(f"{mode.capitalize()} F1-Score Across Epochs")
  plt.xlabel("Epoch")
  plt.ylabel("F1-Score")
  plt.grid(True, linestyle="--", alpha=0.6)
  plt.legend(title="Hyperparameters", bbox_to_anchor=(1.05, 1), loc="upper left")
  plt.tight_layout()

  # Save the line plot.
  os.makedirs(plot_path, exist_ok=True)
  plt.savefig(plot_path, dpi=300)
  plt.show()

def training_plot(i=0, few_shot=False, plot_path="/content/drive/MyDrive/bachelor_thesis/results/performance.png"):
  # Create a line plot for training F1.
  metric_plot("training", "train_f1", i, few_shot, plot_path)

def validation_plot(i=0, few_shot=False, plot_path="/content/drive/MyDrive/bachelor_thesis/results/performance.png"):
  # Create a line plot for validation F1.
  metric_plot("validation", "val_f1", i, few_shot, plot_path)

In [ ]:
# Run the functions once on the few-shot model.
print("--- Few-Shot Model ---")
training_plot(i=0, few_shot=True, plot_path="/content/drive/MyDrive/bachelor_thesis/results/few_shot/training_performance_few_shot.png")
validation_plot(i=0, few_shot=True, plot_path="/content/drive/MyDrive/bachelor_thesis/results/few_shot/validation_performance_few_shot.png")

# Iterate through each version of the transformer model.
print("--- Transformer Model ---")
for i in range(6):
  print(f"--- Version {i + 1} ---")
  training_plot(i=i, few_shot=False, plot_path=f"/content/drive/MyDrive/bachelor_thesis/results/v{i + 1}/training_performance_{i + 1}.png")
  validation_plot(i=i, few_shot=False, plot_path=f"/content/drive/MyDrive/bachelor_thesis/results/v{i + 1}/validation_performance_{i + 1}.png")

### Ablation Study

Since this project aims to serve a medical purpose, an ablation study is conducted in order to determine which classification threshold can be the most suitable for this model, based on two general ideas:
1. **Lowering the classification threshold** makes the model more sensitive to positive examples, which can come in handy for early detection of Parkinson's disease, although the model also becomes more vulnerable to false positives.
2. **Increasing the classification threshold** requires more confidence towards positive examples, reducing the risk of false positives, although the model may not be able to detect early or mild symptoms of Parkinson's disease.

In [ ]:
def ablation_plot(i=0, few_shot=False, plot_path="/content/drive/MyDrive/bachelor_thesis/results/ablation_study.png"):
  if few_shot:
    # Load the metrics associated to the few-shot model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/few_shot/ablation_study_1_fold_{k + 1}.csv") for k in range(5)]
  else:
    # Load the metrics associated to one version of the transformer model.
    files = [pd.read_csv(f"/content/drive/MyDrive/bachelor_thesis/metrics/v{i + 1}/ablation_study_{i + 1}_fold_{k + 1}.csv") for k in range(5)]
  all_data = pd.concat(files, ignore_index=True)

  # Use the melt function to plot precision, recall and F1 together with respect to the identifier threshold.
  melted_data = pd.melt(all_data,
                        id_vars=["threshold"],
                        value_vars=["precision", "recall", "f1"],
                        var_name="Metric",
                        value_name="Score")

  # Create the line plot, using the errorbar="sd" argument to plot mean and standard deviation.
  plt.figure(figsize=(9, 5))
  sns.lineplot(data=melted_data,
               x="threshold",
               y="Score",
               hue="Metric",
               style="Metric",
               errorbar="sd")
  plt.title("Ablation Study - Performance vs Classification Threshold")
  plt.xlabel("Classification Threshold")
  plt.ylabel("Score")
  plt.grid(True, linestyle="--", alpha=0.6)
  plt.tight_layout()

  # Save the line plot.
  os.makedirs(plot_path, exist_ok=True)
  plt.savefig(plot_path, dpi=300)
  plt.show()

In [ ]:
# Run the function once on the few-shot model.
print("--- Few-Shot Model ---")
ablation_plot(i=0, few_shot=True, plot_path="/content/drive/MyDrive/bachelor_thesis/results/few_shot/ablation_study_few_shot.png")

# Iterate through each version of the transformer model.
print("--- Transformer Model ---")
for i in range(6):
  print(f"--- Version {i + 1} ---")
  ablation_plot(i=i, few_shot=False, plot_path=f"/content/drive/MyDrive/bachelor_thesis/results/v{i + 1}/ablation_study_{i + 1}.png")